In [1]:
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pandas as pd
import numpy as np
import seaborn as sns
import gseapy as gp
from gseapy import gseaplot

In [2]:
import scanpy as sc

adata = sc.read_h5ad(r"D:\Dom\Psoriasis project\4th year data\Second Round Data\Xenium outputs - second round\Resegmented Xenium Outputs\ResolVI_subclusters.h5ad")
#only considering dorsal epi/dermis
adata = adata[adata.obs['compartment'].str.contains('orsal')]
adata.obs.compartment.unique()

['dorsal_dermis', 'dorsal_epidermis']
Categories (2, object): ['dorsal_dermis', 'dorsal_epidermis']

In [3]:
from pathlib import Path
results_path = Path(r"C:\Users\dbuxton\OneDrive\Desktop\biochem\Year 4\Thesis\results figures or slides\DESEQ_results")

In [4]:
import decoupler as dc

#making pseudobulk objects for the epidermis
adata_epi = adata[adata.obs['compartment'].str.contains('_epidermis')].copy()

##Filtering genes not expressed by at least 1% of cells
min_cells = int(0.01 * adata_epi.n_obs)
sc.pp.filter_genes(adata_epi, min_cells=min_cells)


big_bulk_epidermis = dc.pp.pseudobulk(
    adata_epi,
    sample_col = 'batch_key', #defines replicates
    groups_col = None, #ignore if using batch_key, add in if using celltypes
    layer = None,
    mode = 'sum',
    empty = True
)


#making pseudobulk objects for the dermis
adata_derm = adata[adata.obs['compartment'].str.contains('_dermis')].copy()

##Filtering genes not expressed by at least 1% of cells
min_cells = int(0.01 * adata_derm.n_obs)
sc.pp.filter_genes(adata_derm, min_cells=min_cells)

big_bulk_dermis = dc.pp.pseudobulk(
    adata_derm,
    sample_col = 'batch_key', #defines replicates
    groups_col = None, #ignore if using batch_key, add in if using celltypes
    layer = None,
    mode = 'sum',
    empty = True
)

d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
#Creating separate objects for males and females

def isolate_sex(adata, sex_str):
    bdata = adata[adata.obs.replicate.str.contains(sex_str)].copy()
    return bdata

bulk_objects = {'big_bulk_epidermis':big_bulk_epidermis, 'big_bulk_dermis':big_bulk_dermis}

bulk_subtypes = {}
for name, obj in bulk_objects.items():
    #bulk_subtypes[name] = obj #can remove this if you don't care about non-sex differences
    for sex in ['F', 'M']:
        bulk_subtypes[f'{name}_{sex}'] = isolate_sex(obj, sex)

for name, obj in bulk_subtypes.items():
    print(name)
    print(obj)

big_bulk_epidermis_F
AnnData object with n_obs × n_vars = 15 × 2144
    obs: 'batch_key', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'region', 'z_level', 'compartment', 'replicate', 'condition', '_scvi_batch', '_scvi_labels', 'highlight', 'sex', 'psbulk_cells', 'psbulk_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mean_log1p_expression', 'n_cells'
    layers: 'psbulk_props'
big_bulk_epidermis_M
AnnData object with n_obs × n_vars = 15 × 2144
    obs: 'batch_key', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'region', 'z_level', 'compartment', 'replicate', 'condition', '_scvi_batch', '_scvi_labels', 'highlight', 'sex', 'psbulk_cells', 'psbulk_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'mean_log1p_expression', 'n_cells'
    layers: 'psbulk_props'
big_bulk_dermis_F
AnnData object with n_obs × n_vars = 15 × 937
    obs: 'batch_key', 'control_p

In [6]:
mechano_genes = pd.read_csv(r"C:\Users\dbuxton\Downloads\panel_psoriasis_subset_with_mouse.csv", index_col = 1)
mechano_genes.index

Index(['CDH1', 'CDH3', 'CLDN1', 'CTNNB1', 'DSG3', 'EZH2', 'POSTN', 'TGFB1',
       'EDN1', 'SPRR3', 'TNC', 'ITGAV', 'CD44', 'HAS2', 'WWTR1', 'YAP1',
       'EPAS1', 'NFE2L2', 'BIRC5', 'CDKN1A', 'KLF5', 'JAK1', 'JAK2', 'STAT3',
       'MTOR', 'HIF1A', 'VEGFA', 'KLF4', 'MRTFA', 'MRTFB', 'SRF', 'PIEZO1',
       'PKD1', 'TRPV1', 'MMP14'],
      dtype='object', name='gene')

In [7]:
mechano_genes.index.to_list()
list = [x.lower().capitalize() for x in mechano_genes.index.to_list()]

In [8]:
gene_panel = pd.read_csv(r"C:\Users\dbuxton\Downloads\XeniumPrimeMouse5Kpan_tissue_pathways_metadata.csv", index_col = 0)
gene_panel.head(1)

,gene_id,num_codewords,num_probesets,protein_name,location,cell_type,cellchat_pathway
gene_name,,,,,,,
A1cf,ENSMUSG00000052595,2,2,APOBEC1 complementation factor (APOBEC1-stimul...,Nucleus,NaN,NaN


In [9]:
list_in5k = []

for i in list:
    if i in gene_panel.index:
        list_in5k.append(i)
    else:
        continue
list_in5k
df_in5k = pd.DataFrame(list_in5k)


In [12]:
import matplotlib.pyplot as plt

for name, bulk in bulk_subtypes.items():
    #make a deseq data set from your anndata
    
    ###avoid recalculating if already done###
    # if Path(results_path/ f'{name}_Ctrl_D3IMQ_DEgenes.csv').exists(): #bit jammy, but if D3 is calced, then so are D7/D10
    #     continue

    dds = DeseqDataSet(counts = bulk.X,
                   metadata = bulk.obs,
                   design_factors = 'condition')
    dds.var = bulk.var


    #run deseq2 on it
    try:
        dds.deseq2()
    except ValueError as e:
        print(f'{e}_{name} failed :(')
        continue

    #get the stats
    for day in ['D7IMQ']:
        print(name.split("_")[3])
        print("="*80)


        stat_res = DeseqStats(dds, contrast = ['condition', day, 'Ctrl']) #pairwise comparison of days with each condition
        stat_res.summary()
        #get diffexp dataframe
        res = stat_res.results_df
        res_file = f'{name}_Ctrl_{day}_DEgenes.csv'
        #res.to_csv(results_path / res_file) #save table with stats of DE genes
        
        #filtering any genes with baseMean <10, choice is somewhat arbitrary
        res = res[res.baseMean >10]
        #then filter all those that are not significant or large enough
        sigs = res[(abs(res.log2FoldChange) > 1)&(res.padj <0.05)] #again, log2fc threshold is bit arbitrary,being stringent so using 1.

        
        
        # #plot CLUSTERMAP for significant genes
        cluster_map_name = f'{name}_Ctrl_{day}__mechano_cluster.png'
        cluster_map_path = results_path / 'clustermaps' / cluster_map_name
        if len(sigs) >2:
            dds.layers['log1p'] = np.log1p(dds.layers['normed_counts'])
            #check for valid genes
            valid_genes = [g for g in df_in5k[0] if g in dds.var_names]

            dds_sigs = dds[:, valid_genes].copy()

            grapher = pd.DataFrame(dds_sigs.layers['log1p'].T,
                        index = dds_sigs.var_names, columns = dds_sigs.obs_names)
            
            clustermap = sns.clustermap(grapher, z_score = 0, cmap = 'RdYlBu_r')
            clustermap.ax_heatmap.tick_params(axis = 'y', labelsize = '20')
            clustermap.ax_heatmap.tick_params(axis = 'x', labelsize = '20')


            clustermap.savefig(cluster_map_path, bbox_inches = 'tight', dpi = 400)
            plt.close(clustermap.fig)
        else:
            print(f'{name} failed at clustermapping')

    
        # #GSEA 


        # ranking = res['stat'].dropna().sort_values(ascending = False)

        # pre_res = gp.prerank(rnk = ranking,
        #                      gene_sets = {'list_in5k': list_in5k},
        #                      seed = 6, permutation_num = 100)
        
        # if name.split("_")[3] == 'F':
        #     sex = 'female'
        # else:
        #     sex = 'male'

        # gseaplot(pre_res.ranking, **pre_res.results['list_in5k'])
        # break

C:\Users\dbuxton\AppData\Local\Temp\ipykernel_21644\801569003.py:10: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts = bulk.X,
Fitting size factors...
... done in 0.00 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 0.35 seconds.

Fitting dispersion trend curve...
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\pydeseq2\dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.03 seconds.

Fitting MAP dispersions...
... done in 0.35 seconds.

Fitting LFCs...
... done in 0.33 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


F


... done in 0.30 seconds.



...

Log2 fold change & Wald test p-value: condition D7IMQ vs Ctrl
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
Aatf     25.835395       -0.729388  0.369166 -1.975773  0.048180  0.221652
Abca1    62.047854        0.193164  0.312309  0.618502  0.536244  0.803249
Abca7    29.491863       -0.241587  0.330997 -0.729878  0.465465  0.744397
Abcc1    15.412003       -0.217666  0.452249 -0.481296  0.630306       NaN
Abcf1    16.961779       -0.946795  0.396932 -2.385282  0.017066  0.122393
...            ...             ...       ...       ...       ...       ...
Zc3h11a  18.306914       -0.599099  0.361146 -1.658884  0.097139  0.302351
Zdhhc8   14.864500       -0.026766  0.474433 -0.056416  0.955011       NaN
Zeb2     27.888959        0.206933  0.319890  0.646889  0.517704  0.785293
Zfp106   26.239420       -0.595341  0.325548 -1.828737  0.067439  0.252048
Zzef1    16.663262        0.206355  0.424298  0.486345  0.626723  0.848613

[937 rows x 6 columns]


C:\Users\dbuxton\AppData\Local\Temp\ipykernel_21644\801569003.py:10: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts = bulk.X,
Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...


Using None as control genes, passed at DeseqDataSet initialization


... done in 0.24 seconds.

Fitting dispersion trend curve...
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\pydeseq2\dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.01 seconds.

Fitting MAP dispersions...
... done in 0.21 seconds.

Fitting LFCs...
... done in 0.21 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...
... done in 0.18 seconds.



M
Log2 fold change & Wald test p-value: condition D7IMQ vs Ctrl
          baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
Aatf     23.440309       -0.567454  0.358563 -1.582579  0.113518  0.309113
Abca1    76.806300        0.100404  0.265141  0.378684  0.704923  0.838214
Abca7    36.265019        0.017276  0.286989  0.060196  0.952000  0.980246
Abcc1    15.211495       -0.189371  0.421571 -0.449204  0.653285  0.815247
Abcf1    17.595919       -0.301472  0.367822 -0.819613  0.412437  0.634570
...            ...             ...       ...       ...       ...       ...
Zc3h11a  23.010033        0.249700  0.365124  0.683878  0.494052  0.692641
Zdhhc8   19.118576       -0.898741  0.361772 -2.484273  0.012982  0.072306
Zeb2     33.662585       -0.103053  0.326897 -0.315246  0.752575  0.863563
Zfp106   30.506359       -0.038728  0.343502 -0.112743  0.910234  0.968037
Zzef1    17.285826        0.770799  0.446733  1.725412  0.084453  0.262029

[937 rows x 6 columns]


In [ ]:
import matplotlib.pyplot as plt

for name, bulk in bulk_subtypes.items():
    #make a deseq data set from your anndata
    
    ###avoid recalculating if already done###
    # if Path(results_path/ f'{name}_Ctrl_D3IMQ_DEgenes.csv').exists(): #bit jammy, but if D3 is calced, then so are D7/D10
    #     continue

    dds = DeseqDataSet(counts = bulk.X,
                   metadata = bulk.obs,
                   design_factors = 'condition')
    dds.var = bulk.var


    #run deseq2 on it
    try:
        dds.deseq2()
    except ValueError as e:
        print(f'{e}_{name} failed :(')
        continue

    #get the stats
    for day in ['D7IMQ']:
        print(name.split("_")[3])
        print("="*80)


        stat_res = DeseqStats(dds, contrast = ['condition', day, 'Ctrl']) #pairwise comparison of days with each condition
        stat_res.summary()
        #get diffexp dataframe
        res = stat_res.results_df
        res_file = f'{name}_Ctrl_{day}_DEgenes.csv'
        #res.to_csv(results_path / res_file) #save table with stats of DE genes
        
        #filtering any genes with baseMean <10, choice is somewhat arbitrary
        res = res[res.baseMean >10]
        #then filter all those that are not significant or large enough
        sigs = res[(abs(res.log2FoldChange) > 1)&(res.padj <0.05)] #again, log2fc threshold is bit arbitrary,being stringent so using 1.

        
        
        # #plot CLUSTERMAP for significant genes
        cluster_map_name = f'{name}_Ctrl_{day}__mechano_cluster.png'
        cluster_map_path = results_path / 'clustermaps' / cluster_map_name
        if len(sigs) >2:
            dds.layers['log1p'] = np.log1p(dds.layers['normed_counts'])
            #check for valid genes
            valid_genes = [g for g in df_in5k[0] if g in dds.var_names]

            dds_sigs = dds[:, valid_genes].copy()

            grapher = pd.DataFrame(dds_sigs.layers['log1p'].T,
                        index = dds_sigs.var_names, columns = dds_sigs.obs_names)
            
            clustermap = sns.clustermap(grapher, z_score = 0, cmap = 'RdYlBu_r')

            clustermap.ax_heatmap.tick_params(axis='y', labelsize=20) # TOGGLE SIZE HERE
    
            clustermap.ax_heatmap.tick_params(axis='x', labelsize=20)

           # clustermap.savefig(cluster_map_path, bbox_inches = 'tight', dpi = 400)
            plt.close(clustermap.fig)
        else:
            print(f'{name} failed at clustermapping')

        # #GSEA 


        # ranking = res['stat'].dropna().sort_values(ascending = False)

        # pre_res = gp.prerank(rnk = ranking,
        #                      gene_sets = {'list_in5k': list_in5k},
        #                      seed = 6, permutation_num = 100)
        
        # if name.split("_")[3] == 'F':
        #     sex = 'female'
        # else:
        #     sex = 'male'

        # gseaplot(pre_res.ranking, **pre_res.results['list_in5k'])
        # break